In [1]:
# ==========================================
# MODELO DE PRODUCCIÓN: Approximate Nearest Neighbors (Annoy)
# Librería nativa de Spotify para Big Data
# ==========================================
import pandas as pd
import numpy as np
import time
import sys
from annoy import AnnoyIndex

# Conectar con el módulo evaluador oficial del equipo
sys.path.append('../src')
try:
    from evaluador import evaluar_modelo_recomendacion
except ImportError:
    print("⚠️ Recuerda tener 'evaluador.py' en la carpeta 'src'.")

# Cargar los datos
ruta_csv = "C:/Users/basti/proyecto_cd/data/processed/spotify_dataset_processed.csv"
df = pd.read_csv(ruta_csv)

features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

In [2]:
print("🌲 Construyendo el bosque hiperdimensional de Annoy...")

dimensiones = len(features)
# 'angular' es la métrica de Annoy equivalente a la similitud del coseno
annoy_model = AnnoyIndex(dimensiones, 'angular') 

# 1. Alimentar el modelo con los vectores (indexados por su posición en el DataFrame)
for i in range(len(df)):
    vector_acustico = df.iloc[i][features].values
    annoy_model.add_item(i, vector_acustico)

# 2. Construir los árboles (10 árboles es el estándar de la industria para balancear velocidad/precisión)
annoy_model.build(10)

print("✅ ¡Modelo Annoy entrenado y listo para producción!")

🌲 Construyendo el bosque hiperdimensional de Annoy...
✅ ¡Modelo Annoy entrenado y listo para producción!


In [3]:
# --- 1. FUNCIÓN DE BÚSQUEDA ---
def buscar_cancion_id(nombre, dataset, artista=None):
    filtro_nombre = dataset['track_name'].str.lower() == nombre.lower()
    resultado = dataset[filtro_nombre]
    if resultado.empty:
        print(f"❌ No se encontró: '{nombre}'")
        return None
    if artista:
        filtro_artista = resultado['artists'].str.contains(artista, case=False, na=False)
        resultado = resultado[filtro_artista]
        if resultado.empty:
            return None
    if len(resultado) > 1:
        print(f"⚠️ Múltiples versiones. Usando la de: {resultado.iloc[0]['artists']}")
    return resultado.iloc[0]['track_id']


# --- 2. MOTOR DE EXPLICABILIDAD ---
def generar_explicacion_acustica(vector_semilla, vector_recomendado, features_list):
    diferencias = np.abs(vector_semilla.values[0] - vector_recomendado.values[0])
    traduccion = {
        'danceability': 'Bailabilidad', 'energy': 'Nivel de Energía', 'loudness': 'Volumen',
        'speechiness': 'Presencia Vocal', 'acousticness': 'Acústica', 'instrumentalness': 'Instrumentalidad',
        'liveness': 'Sensación de Vivo', 'valence': 'Positividad Musical', 'tempo': 'Ritmo'
    }
    dif_dict = {features_list[i]: diferencias[i] for i in range(len(features_list))}
    top_3 = sorted(dif_dict.items(), key=lambda item: item[1])[:3]
    vars_explicacion = [traduccion[var[0]] for var in top_3]
    return f"💡 Recomendada por su similitud en: {vars_explicacion[0]}, {vars_explicacion[1]} y {vars_explicacion[2]}."


# --- 3. MOTOR PRINCIPAL ANNOY AUTO-EVALUABLE ---
def recomendar_por_nombre_annoy_y_evaluar(nombre_cancion, modelo_annoy, dataframe, features_list, artista=None, top_k=5):
    # ⏱️ 1. Iniciar cronómetro de latencia real
    inicio_tiempo = time.time()
    
    # 2. Buscar ID y vector semilla
    track_id = buscar_cancion_id(nombre_cancion, dataframe, artista)
    if not track_id:
        return None, None
        
    idx_semilla = dataframe.index[dataframe['track_id'] == track_id].tolist()[0]
    vector_semilla = dataframe.iloc[[idx_semilla]][features_list]
    
    nombre_semilla = dataframe.iloc[idx_semilla]['track_name']
    artista_semilla = dataframe.iloc[idx_semilla]['artists']
    print(f"\n🔍 Generando recomendaciones ultrarrápidas para: '{nombre_semilla}' de {artista_semilla}")
    print("-" * 70)

    # 3. La Búsqueda Annoy (K+1 para excluir la semilla)
    # include_distances=True nos devuelve la distancia angular de Annoy
    indices_vecinos, distancias = modelo_annoy.get_nns_by_item(idx_semilla, top_k + 1, include_distances=True)

    lista_recomendaciones = []
    
    # 4. Procesar resultados (saltando el índice 0 que es la semilla misma)
    for i in range(1, len(indices_vecinos)):
        idx_vecino = indices_vecinos[i]
        dist_angular = distancias[i]
        
        track_rec = dataframe.iloc[idx_vecino]
        vector_rec = dataframe.iloc[[idx_vecino]][features_list]
        
        explicacion = generar_explicacion_acustica(vector_semilla, vector_rec, features_list)
        
        print(f"{i}. 🎵 {track_rec['track_name']} - {track_rec['artists']}")
        print(f"   {explicacion}")
        print(f"   (Distancia Angular: {dist_angular:.4f})\n")
        
        lista_recomendaciones.append({
            'track_name': track_rec['track_name'],
            'artists': track_rec['artists']
        })
        
    print("-" * 70)
    
    # 5. Extraer vectores puros para el evaluador oficial
    vectores_recomendados = dataframe.iloc[indices_vecinos[1:]][features_list]
    
    # ⏱️ 6. Detener cronómetro
    fin_tiempo = time.time()
    
    # 7. Auditar el modelo
    try:
        evaluar_modelo_recomendacion("Annoy (Producción - Spotify API)", vector_semilla, vectores_recomendados, inicio_tiempo, fin_tiempo)
    except NameError:
        print("⚠️ Evaluador no disponible en memoria.")
        
    return lista_recomendaciones, vectores_recomendados

In [4]:
def iniciar_app_annoy(df, modelo_annoy, features_list):
    print("=" * 65)
    print("🎧 SISTEMA DE RECOMENDACIÓN EN PRODUCCIÓN (MOTOR ANNOY) 🎧")
    print("=" * 65)
    
    while True:
        print("\n" + "-" * 65)
        entrada_cancion = input("🎵 Ingresa el nombre de la canción (o 'salir'): ").strip()
        
        if entrada_cancion.lower() == 'salir':
            print("\n¡Gracias por probar el sistema en producción! Cerrando... 👋")
            break
            
        entrada_artista = input("🎤 Ingresa el artista (Enter para omitir): ").strip()
        entrada_artista = None if entrada_artista == "" else entrada_artista
            
        print("\nProcesando solicitud en la arquitectura de Big Data...")
        
        # Ejecutar el motor auto-evaluado
        recomendar_por_nombre_annoy_y_evaluar(
            nombre_cancion=entrada_cancion, 
            modelo_annoy=modelo_annoy, 
            dataframe=df, 
            features_list=features_list, 
            artista=entrada_artista
        )

# ¡Iniciar la aplicación!
iniciar_app_annoy(df, annoy_model, features)

🎧 SISTEMA DE RECOMENDACIÓN EN PRODUCCIÓN (MOTOR ANNOY) 🎧

-----------------------------------------------------------------


🎵 Ingresa el nombre de la canción (o 'salir'):  Beat it
🎤 Ingresa el artista (Enter para omitir):  Michael Jackson



Procesando solicitud en la arquitectura de Big Data...
❌ No se encontró: 'Beat it'

-----------------------------------------------------------------


🎵 Ingresa el nombre de la canción (o 'salir'):  The Lady In My Life
🎤 Ingresa el artista (Enter para omitir):  



Procesando solicitud en la arquitectura de Big Data...
❌ No se encontró: 'The Lady In My Life'

-----------------------------------------------------------------


...


Procesando solicitud en la arquitectura de Big Data...

🔍 Generando recomendaciones ultrarrápidas para: 'Eenie Meenie' de Sean Kingston;Justin Bieber
----------------------------------------------------------------------
1. 🎵 Feliz Aniversário - Ronaldo E Os Barcellos
   💡 Recomendada por su similitud en: Instrumentalidad, Presencia Vocal y Ritmo.
   (Distancia Angular: 0.0415)

2. 🎵 Friends - levitation room
   💡 Recomendada por su similitud en: Instrumentalidad, Presencia Vocal y Sensación de Vivo.
   (Distancia Angular: 0.0435)

3. 🎵 What Makes You Beautiful - Kids Rock Kidz
   💡 Recomendada por su similitud en: Instrumentalidad, Presencia Vocal y Positividad Musical.
   (Distancia Angular: 0.0512)

4. 🎵 Superlove - Royal Republic
   💡 Recomendada por su similitud en: Presencia Vocal, Instrumentalidad y Sensación de Vivo.
   (Distancia Angular: 0.0517)

5. 🎵 春泥棒 - ヨルシカ
   💡 Recomendada por su similitud en: Instrumentalidad, Positividad Musical y Sensación de Vivo.
   (Distancia Ang

🎵 Ingresa el nombre de la canción (o 'salir'):  Cold Water
🎤 Ingresa el artista (Enter para omitir):  Justin Bieber



Procesando solicitud en la arquitectura de Big Data...

🔍 Generando recomendaciones ultrarrápidas para: 'Cold Water' de Major Lazer;Justin Bieber;MØ
----------------------------------------------------------------------
1. 🎵 I Will Follow - Chris Tomlin
   💡 Recomendada por su similitud en: Instrumentalidad, Acústica y Ritmo.
   (Distancia Angular: 0.0374)

2. 🎵 宝物 - WANIMA
   💡 Recomendada por su similitud en: Bailabilidad, Instrumentalidad y Sensación de Vivo.
   (Distancia Angular: 0.0502)

3. 🎵 Cleopatra - ConeCrewDiretoria
   💡 Recomendada por su similitud en: Instrumentalidad, Ritmo y Acústica.
   (Distancia Angular: 0.0505)

4. 🎵 Nobody But Me - The Human Beinz
   💡 Recomendada por su similitud en: Instrumentalidad, Acústica y Presencia Vocal.
   (Distancia Angular: 0.0531)

5. 🎵 Maximum - Yxng Bane
   💡 Recomendada por su similitud en: Instrumentalidad, Positividad Musical y Presencia Vocal.
   (Distancia Angular: 0.0553)

------------------------------------------------------

🎵 Ingresa el nombre de la canción (o 'salir'):  salir



¡Gracias por probar el sistema en producción! Cerrando... 👋
